# Day 5 — Anthropic (Claude) edition

## A full business solution: a company Brochure generator

We extend the Day 1 project. Given a company name and website, we:
1. scrape the landing page's links,
2. have Claude decide (in **structured JSON**) which links matter,
3. scrape those pages and have Claude write a brochure,
4. then stream the brochure with a typewriter effect.

Two Anthropic-specific things to watch for in this notebook:
- **Getting JSON out of Claude.** Anthropic has no `response_format={"type":"json_object"}`. The idiomatic trick is **assistant prefill**: we seed the assistant's reply with an opening `{`, which forces Claude to continue as raw JSON. We then prepend that `{` back before parsing.
- **Streaming** uses `with client.messages.stream(...) as stream:` and iterating `stream.text_stream`.

In [1]:
# imports
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
import anthropic

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('ANTHROPIC_API_KEY')

if api_key and api_key.startswith('sk-ant-') and len(api_key) > 10:
    print("API key looks good so far")
else:
    print("There might be a problem with your Anthropic API key? Check it starts sk-ant-")

# Cheap, fast model for the structural link-selection task
MODEL = 'claude-haiku-4-5'
client = anthropic.Anthropic()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## Step 1: have Claude figure out which links are relevant

We ask Claude to read the links and respond in structured JSON — deciding which are relevant and turning relative links like `/about` into full URLs. We use "one-shot prompting": we show one example of the desired JSON shape.

This is a great LLM use case — coding it by hand would be painful.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/

### The JSON prefill trick

Notice the last message in the list: `{"role": "assistant", "content": "{"}`. By prefilling the assistant turn with an opening brace, Claude is forced to continue from there as raw JSON (no markdown fences, no preamble). We then stick the `{` back on the front before `json.loads`.

In [ ]:
def select_relevant_links(url):
    response = client.messages.create(
        model=MODEL,
        max_tokens=1000,
        system=link_system_prompt,
        messages=[
            {"role": "user", "content": get_links_user_prompt(url)},
            {"role": "assistant", "content": "{"},  # prefill -> forces JSON
        ],
    )
    result = "{" + response.content[0].text  # prepend the prefilled brace
    links = json.loads(result)
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [17]:
def capital_city():
    response = client.messages.create(
        model = "claude-haiku-4-5",
        max_tokens=1024,
        system= "Repsond in JSON Format only",
        messages=[
            {"role" : "user", "content":"What is the capital of France?"},
            {"role":"assistant","content":"{"}
        ],
    )
    result = "{" + response.content[0].text
    print(json.loads(result))

In [18]:
capital_city()

{'question': 'What is the capital of France?', 'answer': 'Paris'}


Same function with a couple of print statements so you can watch it work:

In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = client.messages.create(
        model=MODEL,
        max_tokens=1000,
        system=link_system_prompt,
        messages=[
            {"role": "user", "content": get_links_user_prompt(url)},
            {"role": "assistant", "content": "{"},
        ],
    )
    result = "{" + response.content[0].text
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [ ]:
select_relevant_links("https://huggingface.co")

## Step 2: make the brochure

Assemble the landing page plus all relevant pages into one big prompt, and have Claude write the brochure.

In [ ]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [ ]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# For a more humorous brochure, swap in this version - shows how easy it is to set 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000]  # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

For the brochure itself we step up to **Claude Sonnet** for higher-quality writing (the link-picking task above stayed on cheap Haiku). Swap the model string if you'd rather keep it all on Haiku.

In [ ]:
def create_brochure(company_name, url):
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=2000,
        system=brochure_system_prompt,
        messages=[
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.content[0].text
    display(Markdown(result))

In [ ]:
create_brochure("HuggingFace", "https://huggingface.co")

## Finally — streaming

With a small change we can stream the result back with the familiar typewriter animation. Anthropic's streaming uses a context manager (`with client.messages.stream(...)`) and you iterate over `stream.text_stream`.

In [ ]:
def stream_brochure(company_name, url):
    display_handle = display(Markdown(""), display_id=True)
    response = ""
    with client.messages.stream(
        model="claude-sonnet-4-6",
        max_tokens=2000,
        system=brochure_system_prompt,
        messages=[
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    ) as stream:
        for text in stream.text_stream:
            response += text
            update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

In [ ]:
# Try the humorous system prompt above, then re-run this:
stream_brochure("HuggingFace", "https://huggingface.co")

## Business applications

We extended Day 1 into a multi-call pipeline that generates a document — arguably your first taste of an **Agentic AI** design pattern (chaining multiple LLM calls). Content generation like this maps onto endless business tasks: marketing copy, product tutorials from a spec, personalized email, and more. Try a proof-of-concept for your own use case.

## Before Week 2

Have a go at the `week1 EXERCISE` notebook for the end-of-week challenge — good practice with frontier APIs before Week 2. I can convert that one to Anthropic too when you get there.